In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn torch tqdm huggingface_hub')
    os.system('pip install -q fasttext')
    print("Setup complete!")


In [2]:
import os
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_finetuned_results'
model_dir = 'models/finetuned/openlid'

models_to_evaluate = [
    "openlid_head_no_rehearsal",
    "openlid_head_with_rehearsal"
]


In [3]:
import json
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import fasttext
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
    "eng": "en", "tam": "ta", "hin": "hi", "ben": "bn", "arb": "ar", "fra": "fr", "deu": "de",
    "jpn": "ja", "nld": "nl", "pol": "pl", "ita": "it", "por": "pt", "tur": "tr", "spa": "es", "ell": "el", "urd": "ur", "bul": "bg", "cmn": "zh", "rus": "ru", "tha": "th", "swh": "sw", "vie": "vi",
    "sinhala": "si", "sanskrit": "sa", "pali": "pi"
}

DATASET_TO_OPENLID_MAP = {
    "eng": "__label__eng_Latn",
    "sin": "__label__sin_Sinh",
    "san": "__label__san_Sinh", 
    "pli": "__label__pli_Sinh", 
    "tam": "__label__tam_Taml",
    "hin": "__label__hin_Deva",
    "ben": "__label__ben_Beng",
    "arb": "__label__ara_Arab", 
    "fra": "__label__fra_Latn",
    "deu": "__label__deu_Latn",
    "jpn": "__label__jpn_Jpan",
    "nld": "__label__nld_Latn",
    "pol": "__label__pol_Latn",
    "ita": "__label__ita_Latn",
    "por": "__label__por_Latn",
    "tur": "__label__tur_Latn",
    "spa": "__label__spa_Latn",
    "ell": "__label__ell_Grek",
    "urd": "__label__urd_Arab",
    "bul": "__label__bul_Cyrl",
    "cmn": "__label__cmn_Hans",
    "rus": "__label__rus_Cyrl",
    "tha": "__label__tha_Thai",
    "swh": "__label__swh_Latn",
    "vie": "__label__vie_Latn"
}

OPENLID_TO_METRIC_MAP = {}
for dataset_key, openlid_label in DATASET_TO_OPENLID_MAP.items():
    metric_val = TARGET_LANGUAGES.get(dataset_key)
    if not metric_val:
        if dataset_key == 'sin': metric_val = 'si'
        if dataset_key == 'san': metric_val = 'sa'
        if dataset_key == 'pli': metric_val = 'pi'
    OPENLID_TO_METRIC_MAP[openlid_label] = metric_val

def load_dataset(file_path):
    print(f"\nLoading {os.path.basename(file_path)}...")
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("label") in TARGET_LANGUAGES:
                records.append(row)
    df = pd.DataFrame(records)
    if not df.empty:
        df["mapped_label"] = df["label"].map(TARGET_LANGUAGES)
        print(f"Loaded {len(df)} rows across {df['label'].nunique()} target languages")
    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(results["true_label"], results["predicted_label"], average="macro", labels=target_labels, zero_division=0)

    print("\n" + "=" * 60)
    print(f"BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 60)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 60)
    print("\nPer-language breakdown:\n")
    print(classification_report(results["true_label"], results["predicted_label"], labels=target_labels, digits=4, zero_division=0))

    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"{model_name}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")

dataset_files = glob.glob(os.path.join(input_dir, "*_integrated.jsonl"))


In [4]:
print("Loading Base OpenLID FastText model...")
fasttext.FastText.eprint = lambda x: None
ft_model = fasttext.load_model('models/pretrained/openlid/openlid-v3.bin')
old_labels = ft_model.get_labels()
hidden_dim = ft_model.get_dimension()

new_labels = []
for target_label in DATASET_TO_OPENLID_MAP.values():
    if target_label not in old_labels:
        new_labels.append(target_label)

all_labels = old_labels + new_labels
num_classes = len(all_labels)

print(f"Total expected classes in PyTorch head: {num_classes}")


Loading Base OpenLID FastText model...
Total expected classes in PyTorch head: 197


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
target_metric_labels = sorted(set([v for k,v in TARGET_LANGUAGES.items()]))

def predict_batch(texts, head, batch_size=256):
    predictions = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        embeddings = [ft_model.get_sentence_vector(str(t).replace('\n', ' ')) for t in batch_texts]
        features = torch.tensor(np.array(embeddings), dtype=torch.float32).to(device)
        with torch.no_grad():
            outputs = head(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend([all_labels[idx] for idx in preds])
    return predictions

for model_name in models_to_evaluate:
    weight_path = os.path.join(model_dir, f"{model_name}.pt")
    if not os.path.exists(weight_path):
        print(f"Skipping {model_name}, weights not found at {weight_path}")
        continue
        
    print(f"\n{'#'*80}")
    print(f"EVALUATING MODEL: {model_name}")
    print(f"{'#'*80}")
    
    head = nn.Linear(hidden_dim, num_classes, bias=False)
    head.load_state_dict(torch.load(weight_path, map_location='cpu', weights_only=True))
    head = head.to(device)
    head.eval()
    
    for file_path in dataset_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df = load_dataset(file_path)
        if df.empty: continue
        
        texts = df["text"].tolist()
        
        # Get predictions (returns __label__ formats)
        raw_preds = predict_batch(texts, head)
        
        # Map OpenLID raw labels back to 2-letter evaluation labels
        # If the model predicts a language not in our map, mark it as 'other' so it fails evaluation correctly
        mapped_preds = [OPENLID_TO_METRIC_MAP.get(p, 'other') for p in raw_preds]
        
        results = df[["text", "label", "source"]].copy()
        results["true_label"] = df["mapped_label"]
        results["predicted_label"] = mapped_preds
        
        evaluate_and_save(results, model_name, dataset_name, target_metric_labels)



################################################################################
EVALUATING MODEL: openlid_head_no_rehearsal
################################################################################

Loading wili-2018_integrated.jsonl...
Loaded 26047 rows across 22 target languages

BENCHMARK RESULTS (openlid_head_no_rehearsal on wili-2018_integrated)
Accuracy:  89.93%
Macro F1:  80.57%

Per-language breakdown:

              precision    recall  f1-score   support

          ar     0.0000    0.0000    0.0000         0
          bg     1.0000    0.9640    0.9817      1000
          bn     1.0000    0.8960    0.9451      1000
          de     0.9823    0.9410    0.9612      1000
          el     1.0000    0.9900    0.9950      1000
          en     0.8591    0.9880    0.9191      1000
          es     0.9968    0.9370    0.9660      1000
          fr     0.9886    0.9540    0.9710      1000
          hi     1.0000    0.9690    0.9843      1000
          it     1.0000    0.9150  